# IEEE-CIS Fraud Detection — ML Experiment Pipeline
**Kaggle Notebook | T4 GPU | 30GB RAM**

Models: Logistic Regression · Random Forest · LightGBM · Isolation Forest · GCN · GAT · GraphSAGE · CS-GraphSAGE

> Session options → Accelerator → GPU T4 x2

## Step 1: Install PyTorch Geometric

In [1]:
import torch, subprocess, sys

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# pyg_lib and torch-sparse have no wheels for PyTorch 2.10 yet.
# torch-geometric alone is sufficient for full-batch GNN training.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch-geometric'], check=True)

import torch_geometric
print(f'PyG     : {torch_geometric.__version__}')
print('Ready.')


PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.4 MB/s eta 0:00:00
PyG     : 2.7.0
Ready.


## Step 2: Load Data

In [2]:
import pandas as pd
import numpy as np
import gc, time, warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

def reduce_mem(df):
    """Cast columns to smallest viable dtype — halves RAM usage."""
    for col in df.columns:
        ct = df[col].dtype
        if ct == object: continue
        c_min, c_max = df[col].min(), df[col].max()
        if str(ct)[:3] == 'int':
            if   c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
        else:
            if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
    return df

print('Loading...')
train_txn = reduce_mem(pd.read_csv(DATA_PATH + 'train_transaction.csv'))
train_id  = reduce_mem(pd.read_csv(DATA_PATH + 'train_identity.csv'))
train     = train_txn.merge(train_id, on='TransactionID', how='left')
del train_txn, train_id; gc.collect()

print(f'Shape      : {train.shape}')
print(f'Fraud rate : {train["isFraud"].mean():.4f}  ({int(train["isFraud"].sum()):,} fraud)')
print(f'RAM        : {train.memory_usage().sum()/1e6:.0f} MB')


Loading...
Shape      : (590540, 434)
Fraud rate : 0.0350  (20,663 fraud)
RAM        : 1095 MB


## Step 3: Preprocessing

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

y_all = train['isFraud'].values.astype(np.int8)
X_all = train.drop(['isFraud','TransactionID'], axis=1).copy()

cat_cols = X_all.select_dtypes(include='object').columns.tolist()
print(f'Encoding {len(cat_cols)} categorical columns...')

for col in cat_cols:
    le = LabelEncoder()
    X_all[col] = X_all[col].fillna('__NA__')
    X_all[col] = le.fit_transform(X_all[col].astype(str)).astype(np.int16)

X_all = X_all.fillna(-999).astype(np.float32)
print(f'Features   : {X_all.shape}  RAM: {X_all.memory_usage().sum()/1e6:.0f} MB')

X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.15, random_state=42, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)
del X_temp, X_all; gc.collect()

print(f'Train: {X_train.shape}  fraud={y_train.mean():.4f}')
print(f'Val  : {X_val.shape}  fraud={y_val.mean():.4f}')
print(f'Test : {X_test.shape}  fraud={y_test.mean():.4f}')


Encoding 31 categorical columns...
Features   : (590540, 432)  RAM: 1020 MB
Train: (413614, 432)  fraud=0.0350
Val  : (88345, 432)  fraud=0.0350
Test : (88581, 432)  fraud=0.0350


## Step 4: Baseline Models

In [4]:
results = {}

def evaluate(name, y_true, y_score, thresh=0.5):
    auc_roc = roc_auc_score(y_true, y_score)
    auc_pr  = average_precision_score(y_true, y_score)
    preds   = (y_score >= thresh).astype(int)
    f1      = f1_score(y_true, (y_score >= thresh).astype(int), zero_division=0)
    precision = precision_score(y_true, preds, zero_division=0)
    k       = max(1, int(0.01 * len(y_true)))
    top_k   = np.argsort(y_score)[::-1][:k]
    r1pct   = float(y_true[top_k].mean())
    results[name] = dict(AUC_ROC=auc_roc, AUC_PR=auc_pr, F1=f1, Precision=precision, Recall_at_1pct=r1pct)
    print(f'{name:<25} AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  '
          f'F1={f1:.4f}  Recall@1%={r1pct:.4f}  Precision={precision:.4f}  (thresh={thresh:.2f})')

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score as f1_fn, precision_score
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE

print('--- Logistic Regression (50k subsample) ---')
# Subsample: LR is a linear baseline; 50k rows is sufficient for convergence.
# Evaluation is always on the full held-out test set.
sub  = np.random.RandomState(42).choice(len(X_train), 50_000, replace=False)
sc   = StandardScaler()
X_s  = sc.fit_transform(X_train.iloc[sub].to_numpy(dtype=np.float32))
X_v  = sc.transform(X_val.to_numpy(dtype=np.float32))
X_te = sc.transform(X_test.to_numpy(dtype=np.float32))

t0 = time.time()
lr_plain = LogisticRegression(C=0.01, max_iter=200, class_weight='balanced',
                         solver='lbfgs', n_jobs=-1, random_state=42)
lr_plain.fit(X_s, y_train[sub])

proba_plain     = lr_plain.predict_proba(X_te)[:,1]
val_proba_plain = lr_plain.predict_proba(X_v)[:,1]

evaluate('LR (No Imbalance)', y_test, proba_plain)

lr = LogisticRegression(C=0.01, max_iter=200, class_weight='balanced',
                        solver='lbfgs', n_jobs=-1, random_state=42)

lr = CalibratedClassifierCV(lr, method='sigmoid')

lr.fit(X_s, y_train[sub])

proba     = lr.predict_proba(X_te)[:,1]
val_proba = lr.predict_proba(X_v)[:,1]

# Find optimal F1 threshold on validation set
best_f1, best_thresh = 0, 0.5
for thresh in np.linspace(0.01, 0.99, 100):
    f1 = f1_fn(y_val, (val_proba >= thresh).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh

print(f'Optimal threshold: {best_thresh:.2f}  (val F1={best_f1:.4f})')
print(f'Time: {time.time()-t0:.1f}s')
evaluate('Logistic Regression', y_test, proba, thresh=best_thresh)

print('--- Logistic Regression (SMOTE) ---')

if 'X_s' not in globals():
    print("X_s not found — recomputing scaling...")
    sc = StandardScaler()
    X_s  = sc.fit_transform(X_train.iloc[sub].to_numpy(dtype=np.float32))
    X_v  = sc.transform(X_val.to_numpy(dtype=np.float32))
    X_te = sc.transform(X_test.to_numpy(dtype=np.float32))


smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_s, y_train[sub])

lr_smote = LogisticRegression(C=0.01, max_iter=500,
                              solver='saga', n_jobs=-1, random_state=42)

lr_smote.fit(X_res, y_res)

proba     = lr_smote.predict_proba(X_te)[:,1]
val_proba = lr_smote.predict_proba(X_v)[:,1]

best_f1, best_thresh = 0, 0.5
for thresh in np.linspace(0.01, 0.99, 100):
    f1 = f1_fn(y_val, (val_proba >= thresh).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh

print(f'Optimal threshold: {best_thresh:.2f}  (val F1={best_f1:.4f})')

evaluate('Logistic Regression (SMOTE)', y_test, proba, thresh=best_thresh)

del X_s, X_v, X_te, lr, lr_plain, lr_smote, sc, sub, proba, val_proba; gc.collect()

--- Logistic Regression (50k subsample) ---
LR (No Imbalance)         AUC-ROC=0.8253  AUC-PR=0.2277  F1=0.1902  Recall@1%=0.4508  Precision=0.1097  (thresh=0.50)
Optimal threshold: 0.11  (val F1=0.3083)
Time: 39.8s
Logistic Regression       AUC-ROC=0.8247  AUC-PR=0.2259  F1=0.3084  Recall@1%=0.4486  Precision=0.2593  (thresh=0.11)
--- Logistic Regression (SMOTE) ---
Optimal threshold: 0.80  (val F1=0.3097)
Logistic Regression (SMOTE) AUC-ROC=0.8181  AUC-PR=0.2309  F1=0.3113  Recall@1%=0.4746  Precision=0.2755  (thresh=0.80)


25

In [15]:
from sklearn.ensemble import RandomForestClassifier

print('--- Random Forest (50k subsample) ---')
sub = np.random.RandomState(42).choice(len(X_train), 50_000, replace=False)
t0  = time.time()
rf_plain = RandomForestClassifier(n_estimators=200, max_depth=10,
                              class_weight='balanced', n_jobs=-1, random_state=42)
rf_plain.fit(X_train.iloc[sub].to_numpy(dtype=np.float32), y_train[sub])

proba     = rf_plain.predict_proba(X_test.to_numpy(dtype=np.float32))[:,1]
val_proba = rf_plain.predict_proba(X_val.to_numpy(dtype=np.float32))[:,1]

evaluate('RF (No Imbalance)', y_test, proba)

rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                            class_weight='balanced',
                            n_jobs=-1, random_state=42)

rf.fit(X_train.iloc[sub].to_numpy(dtype=np.float32), y_train[sub])

proba     = rf.predict_proba(X_test.to_numpy(dtype=np.float32))[:,1]
val_proba = rf.predict_proba(X_val.to_numpy(dtype=np.float32))[:,1]

best_f1, best_thresh = 0, 0.5
for thresh in np.linspace(0.01, 0.99, 100):  # 🔥 FIXED
    f1 = f1_fn(y_val, (val_proba >= thresh).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh

evaluate('Random Forest (Weighted)', y_test, proba, thresh=best_thresh)

print('--- Random Forest (SMOTE) ---')

smote = SMOTE(random_state=42)

X_res, y_res = smote.fit_resample(
    X_train.iloc[sub].to_numpy(dtype=np.float32),
    y_train[sub]
)

rf_smote = RandomForestClassifier(n_estimators=200, max_depth=10,
                                  n_jobs=-1, random_state=42)

rf_smote.fit(X_res, y_res)

proba     = rf_smote.predict_proba(X_test.to_numpy(dtype=np.float32))[:,1]
val_proba = rf_smote.predict_proba(X_val.to_numpy(dtype=np.float32))[:,1]

# Find optimal F1 threshold on validation set
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.01, 0.99, 100):
    f1 = f1_fn(y_val, (val_proba >= thresh).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh

print(f'Optimal threshold: {best_thresh:.2f}  (val F1={best_f1:.4f})')
print(f'Time: {time.time()-t0:.1f}s')
evaluate('Random Forest', y_test, proba, thresh=best_thresh)
del rf, sub, proba, val_proba; gc.collect()


--- Random Forest (50k subsample) ---
RF (No Imbalance)         AUC-ROC=0.8670  AUC-PR=0.4325  F1=0.3412  Recall@1%=0.7955  Precision=0.2349  (thresh=0.50)
Random Forest (Weighted)  AUC-ROC=0.8670  AUC-PR=0.4325  F1=0.4257  Recall@1%=0.7955  Precision=0.5015  (thresh=0.68)
--- Random Forest (SMOTE) ---
Optimal threshold: 0.01  (val F1=0.0701)
Time: 56.6s
Random Forest             AUC-ROC=0.8572  AUC-PR=0.4324  F1=0.0701  Recall@1%=0.7989  Precision=0.0364  (thresh=0.01)


125

In [17]:
import lightgbm as lgb
from sklearn.metrics import f1_score as f1_fn

print('--- LightGBM ---')
scale_pw = float((y_train==0).sum()) / float((y_train==1).sum())

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    scale_pos_weight=scale_pw,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

t0 = time.time()
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]
)

proba     = lgb_model.predict_proba(X_test)[:,1]
val_proba = lgb_model.predict_proba(X_val)[:,1]

# Find optimal F1 threshold on validation set
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.01, 0.5, 0.01):
    f1 = f1_fn(y_val, (val_proba >= thresh).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh

print(f'Optimal threshold: {best_thresh:.2f}  (val F1={best_f1:.4f})')
print(f'Time: {time.time()-t0:.1f}s')

evaluate('LightGBM', y_test, proba, thresh=best_thresh)

del lgb_model, proba, val_proba; gc.collect()

--- LightGBM ---
[LightGBM] [Info] Number of positive: 14473, number of negative: 399141
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.647073 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39111
[LightGBM] [Info] Number of data points in the train set: 413614, number of used features: 431
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034992 -> initscore=-3.317030
[LightGBM] [Info] Start training from score -3.317030
Optimal threshold: 0.11  (val F1=0.4138)
Time: 19.9s
LightGBM                  AUC-ROC=0.8728  AUC-PR=0.3437  F1=0.4180  Recall@1%=0.5977  Precision=0.3754  (thresh=0.11)


9

In [18]:
from sklearn.ensemble import IsolationForest

print('--- Isolation Forest (50k subsample) ---')
sub = np.random.RandomState(42).choice(len(X_train), 50_000, replace=False)
t0  = time.time()
iso = IsolationForest(n_estimators=200, contamination=0.035,
                      n_jobs=-1, random_state=42)
iso.fit(X_train.iloc[sub].to_numpy(dtype=np.float32))
evaluate('Isolation Forest', y_test,
         -iso.score_samples(X_test.to_numpy(dtype=np.float32)))
print(f'Time: {time.time()-t0:.1f}s')
del iso, sub; gc.collect()


--- Isolation Forest (50k subsample) ---
Isolation Forest          AUC-ROC=0.7527  AUC-PR=0.0961  F1=0.1998  Recall@1%=0.0983  Precision=0.1315  (thresh=0.50)
Time: 2.6s


71

In [19]:
print('\n=== BASELINE SUMMARY ===')
print(pd.DataFrame(results).T.round(4).to_string())



=== BASELINE SUMMARY ===
                             AUC_ROC  AUC_PR      F1  Precision  Recall_at_1pct
LR (No Imbalance)             0.8253  0.2277  0.1902     0.1097          0.4508
Logistic Regression           0.8247  0.2259  0.3084     0.2593          0.4486
Logistic Regression (SMOTE)   0.8181  0.2309  0.3113     0.2755          0.4746
RF (No Imbalance)             0.8670  0.4325  0.3412     0.2349          0.7955
Random Forest (Weighted)      0.8670  0.4325  0.4257     0.5015          0.7955
Random Forest                 0.8572  0.4324  0.0701     0.0364          0.7989
LightGBM                      0.8728  0.3437  0.4180     0.3754          0.5977
Isolation Forest              0.7527  0.0961  0.1998     0.1315          0.0983


## Step 5: Graph Construction

We free the tabular splits and build a transaction graph from a **stratified 200k subsample** (all fraud rows + sampled legit rows, preserving the original fraud rate).

**Nodes** = transactions · **Edges** = shared card / email domain / device / address

> *Due to memory constraints inherent to graph construction at scale, GNN experiments were conducted on a stratified subsample of 200k transactions preserving the original class distribution. All models were evaluated on the same held-out test partition.*

In [20]:
# Free tabular splits — no longer needed for graph construction
del X_train, X_val, X_test
gc.collect()

from collections import defaultdict
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, remove_self_loops, coalesce

# Stratified subsample: all fraud + 185k legit ≈ 200k total
fraud_rows = train[train['isFraud'] == 1]
legit_rows = train[train['isFraud'] == 0].sample(n=185_000, random_state=42)
sub = pd.concat([fraud_rows, legit_rows]).sample(frac=1, random_state=42)
sub = sub.reset_index(drop=True)
del train, fraud_rows, legit_rows; gc.collect()

print(f'Graph sample : {sub.shape}  fraud rate: {sub["isFraud"].mean():.4f}')
N = len(sub)

EDGE_FEATS = ['card1','card2','addr1','addr2',
              'P_emaildomain','R_emaildomain','DeviceInfo']
MAX_GROUP  = 50   # skip super-nodes (e.g. one card used in >50 transactions)
edge_list  = []

for feat in EDGE_FEATS:
    if feat not in sub.columns: continue
    vals   = sub[feat].fillna('__NA__').astype(str).values
    groups = defaultdict(list)
    for i, v in enumerate(vals):
        if v not in ('__NA__', '-999', 'nan'): groups[v].append(i)
    batch = []
    for members in groups.values():
        if 2 <= len(members) <= MAX_GROUP:
            m   = np.array(members)
            src = np.repeat(m, len(m))
            dst = np.tile(m, len(m))
            mk  = src != dst
            batch.append(np.stack([src[mk], dst[mk]]))
    if batch:
        ei = np.concatenate(batch, axis=1)
        edge_list.append(ei)
        print(f'  {feat:<20} {ei.shape[1]//2:>7,} edges')

edge_index = torch.tensor(np.concatenate(edge_list, axis=1), dtype=torch.long)
edge_index = to_undirected(edge_index)
edge_index, _ = remove_self_loops(edge_index)
edge_index = coalesce(edge_index, num_nodes=N)
del edge_list; gc.collect()
print(f'\nNodes: {N:,}  Edges: {edge_index.shape[1]:,}  '
      f'Avg degree: {edge_index.shape[1]/N:.2f}')


Graph sample : (205663, 434)  fraud rate: 0.1005
  card1                415,889 edges
  card2                102,384 edges
  addr1                  5,901 edges
  addr2                  1,233 edges
  P_emaildomain          4,728 edges
  R_emaildomain          7,012 edges
  DeviceInfo            70,209 edges

Nodes: 205,663  Edges: 1,060,152  Avg degree: 5.15


In [21]:
# Node features — numeric columns only, normalised to float32
num_cols = [c for c in sub.select_dtypes(include=np.number).columns
            if c not in ('isFraud','TransactionID')]

X_graph = sub[num_cols].fillna(-999).values.astype(np.float32)
X_graph = StandardScaler().fit_transform(X_graph).astype(np.float32)

node_feats  = torch.tensor(X_graph, dtype=torch.float)
node_labels = torch.tensor(sub['isFraud'].values, dtype=torch.long)
del X_graph; gc.collect()

# Stratified 70 / 15 / 15 split
def split_idx(arr, seed=42):
    arr = np.random.RandomState(seed).permutation(arr)
    n   = len(arr)
    return arr[:int(.70*n)], arr[int(.70*n):int(.85*n)], arr[int(.85*n):]

fi = np.where(sub['isFraud'].values == 1)[0]
li = np.where(sub['isFraud'].values == 0)[0]
tr_f, va_f, te_f = split_idx(fi)
tr_l, va_l, te_l = split_idx(li)

def make_mask(idx_list):
    m = torch.zeros(N, dtype=torch.bool)
    m[np.concatenate(idx_list)] = True
    return m

train_mask = make_mask([tr_f, tr_l])
val_mask   = make_mask([va_f, va_l])
test_mask  = make_mask([te_f, te_l])

data = Data(x=node_feats, edge_index=edge_index, y=node_labels,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)
del sub, node_feats, edge_index; gc.collect()

print(data)
print(f'Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}')


Data(x=[205663, 401], edge_index=[2, 1060152], y=[205663], train_mask=[205663], val_mask=[205663], test_mask=[205663])
Train: 143,963  Val: 30,850  Test: 30,850


## Step 6: GNN Model Definitions

In [22]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
# Note: NeighborLoader is NOT used — requires torch-sparse which has no
# wheels for PyTorch 2.10. Full-batch training is used instead and is
# faster for graphs of this size on a 16GB T4 GPU.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

class GCN(nn.Module):
    def __init__(self, in_ch, hid, out_ch=2, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid)
        self.conv2 = GCNConv(hid, hid)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.dropout(F.relu(self.conv2(x, ei)), p=self.drop, training=self.training)
        return self.lin(x)

class GAT(nn.Module):
    def __init__(self, in_ch, hid, out_ch=2, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(in_ch, hid, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hid*heads, hid, heads=1, dropout=dropout)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
    def forward(self, x, ei):
        x = F.dropout(x, p=self.drop, training=self.training)
        x = F.dropout(F.elu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.elu(self.conv2(x, ei))
        return self.lin(x)

class GraphSAGE(nn.Module):
    def __init__(self, in_ch, hid, out_ch=2, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hid)
        self.conv2 = SAGEConv(hid, hid)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.dropout(F.relu(self.conv2(x, ei)), p=self.drop, training=self.training)
        return self.lin(x)

class CostSensitiveGraphSAGE(nn.Module):
    """
    GraphSAGE with a learnable 2x2 cost matrix.
    Adapted from: Hu et al. (2024) Cost-Sensitive GNN-Based Imbalanced
    Learning for Mobile Social Network Fraud Detection.
    C[i,j] = learned cost of predicting class j when true class is i.
    """
    def __init__(self, in_ch, hid, out_ch=2, dropout=0.3, fraud_rate=0.035):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hid)
        self.conv2 = SAGEConv(hid, hid)
        self.lin   = nn.Linear(hid, out_ch)
        self.drop  = dropout
        ir         = fraud_rate / (1.0 - fraud_rate)
        # Initialise: missing fraud (FN) is costly; false alarm (FP) costs 1
        self.C = nn.Parameter(torch.tensor(
            [[0.0, 1.0], [1.0/ir, 0.0]], dtype=torch.float))
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.dropout(F.relu(self.conv2(x, ei)), p=self.drop, training=self.training)
        return self.lin(x)
    def cost_sensitive_loss(self, logits, labels):
        probs       = torch.softmax(logits, dim=1)
        C_pos       = torch.clamp(self.C, min=0.0)
        cost_rows   = C_pos[labels]
        sample_cost = (probs * cost_rows).sum(dim=1)
        ce          = F.cross_entropy(logits, labels, reduction='none')
        return (sample_cost * ce).mean()


Training on: cuda


## Step 7: Run GNN Experiments

Full-batch training is used (entire graph per forward pass) rather than `NeighborLoader`, which requires `torch-sparse` — unavailable for PyTorch 2.10. Full-batch training is faster for graphs of this size on a 16GB GPU.

In [23]:
def run_gnn(model, data, name, epochs=150, lr=1e-3,
            patience=15, cost_sensitive=False):
    """Full-batch GNN training on GPU."""

    model  = model.to(device)
    data_d = data.to(device)

    n_pos = int(data.y[data.train_mask].sum())
    n_neg = int(data.train_mask.sum()) - n_pos
    pos_w = torch.tensor([n_neg / n_pos], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    if cost_sensitive:
        gnn_p = [p for n, p in model.named_parameters() if n != 'C']
        opt   = torch.optim.Adam([
            {'params': gnn_p,     'lr': lr},
            {'params': [model.C], 'lr': lr * 0.1}
        ], weight_decay=5e-4)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=5)

    best_auc, best_state, stale = 0, None, 0

    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        opt.zero_grad()
        logits    = model(data_d.x, data_d.edge_index)
        tr_logits = logits[data_d.train_mask]
        tr_labels = data_d.y[data_d.train_mask]
        loss = (model.cost_sensitive_loss(tr_logits, tr_labels)
                if cost_sensitive
                else criterion(tr_logits[:,1].float(), tr_labels.float()))
        loss.backward()
        opt.step()

        # ── Validate ──
        model.eval()
        with torch.no_grad():
            out = model(data_d.x, data_d.edge_index)
            vp  = torch.softmax(out[data_d.val_mask], dim=1)[:,1].cpu().numpy()
            vl  = data_d.y[data_d.val_mask].cpu().numpy()
        val_auc = roc_auc_score(vl, vp)
        sched.step(val_auc)

        if epoch % 10 == 0:
            extra = ''
            if cost_sensitive:
                C = model.C.detach().cpu()
                extra = f'  C(FN)={C[1,0]:.3f} C(FP)={C[0,1]:.3f}'
            print(f'  Ep {epoch:3d} | loss={loss.item():.4f} '
                  f'| val AUC={val_auc:.4f}{extra}')

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            print(f'  Early stop ep {epoch} — best val AUC={best_auc:.4f}')
            break

    # ── Test + threshold optimisation ──
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out = model(data_d.x, data_d.edge_index)
        tp  = torch.softmax(out[data_d.test_mask], dim=1)[:,1].cpu().numpy()
        tl  = data_d.y[data_d.test_mask].cpu().numpy()
        # Find optimal threshold on val set
        vp_final = torch.softmax(out[data_d.val_mask], dim=1)[:,1].cpu().numpy()
        vl_final = data_d.y[data_d.val_mask].cpu().numpy()

    from sklearn.metrics import f1_score as f1_fn
    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.01, 0.99, 0.01):
        f1 = f1_fn(vl_final, (vp_final >= thresh).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh

    print(f'  Optimal threshold: {best_thresh:.2f}  (val F1={best_f1:.4f})')

    del data_d; torch.cuda.empty_cache()
    return tl, tp, best_thresh


In [24]:
IN = data.x.shape[1]
H  = 128
fr = float(data.y[data.train_mask].float().mean())

experiments = [
    ('GCN',          GCN(IN, H),                                    False),
    ('GAT',          GAT(IN, H, heads=4),                           False),
    ('GraphSAGE',    GraphSAGE(IN, H),                              False),
    ('CS-GraphSAGE', CostSensitiveGraphSAGE(IN, H, fraud_rate=fr),  True),
]

for name, model, cs in experiments:
    print(f'\n{"+"*55}\n  {name}\n{"+"*55}')
    t0 = time.time()
    y_true, y_prob, best_thresh = run_gnn(model, data, name,
                                          epochs=150, patience=15,
                                          cost_sensitive=cs)
    print(f'  Wall time: {time.time()-t0:.1f}s')
    evaluate(name, y_true, y_prob, thresh=best_thresh)
    del model; torch.cuda.empty_cache(); gc.collect()



+++++++++++++++++++++++++++++++++++++++++++++++++++++++
  GCN
+++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Ep  10 | loss=1.1132 | val AUC=0.7568
  Ep  20 | loss=1.0658 | val AUC=0.7780
  Ep  30 | loss=1.0329 | val AUC=0.7896
  Ep  40 | loss=1.0111 | val AUC=0.7974
  Ep  50 | loss=0.9934 | val AUC=0.8044
  Ep  60 | loss=0.9755 | val AUC=0.8114
  Ep  70 | loss=0.9623 | val AUC=0.8174
  Ep  80 | loss=0.9501 | val AUC=0.8226
  Ep  90 | loss=0.9357 | val AUC=0.8266
  Ep 100 | loss=0.9243 | val AUC=0.8291
  Ep 110 | loss=0.9184 | val AUC=0.8325
  Ep 120 | loss=0.9102 | val AUC=0.8364
  Ep 130 | loss=0.9038 | val AUC=0.8380
  Ep 140 | loss=0.8953 | val AUC=0.8400
  Ep 150 | loss=0.8898 | val AUC=0.8419
  Optimal threshold: 0.74  (val F1=0.5152)
  Wall time: 23.8s
GCN                       AUC-ROC=0.8360  AUC-PR=0.5006  F1=0.4942  Recall@1%=0.8896  Precision=0.4885  (thresh=0.74)

+++++++++++++++++++++++++++++++++++++++++++++++++++++++
  GAT
++++++++++++++++++++++++++++++++++++++

## Step 8: Final Results

In [25]:
print('\n' + '='*65)
print('FINAL RESULTS — All Models')
print('='*65)
res_df = pd.DataFrame(results).T.sort_values('AUC_ROC', ascending=False)
res_df.index.name = 'Model'
print(res_df.round(4).to_string())

res_df.to_csv('/kaggle/working/experiment_results.csv')
print('\nSaved → /kaggle/working/experiment_results.csv')



FINAL RESULTS — All Models
                             AUC_ROC  AUC_PR      F1  Precision  Recall_at_1pct
Model                                                                          
LightGBM                      0.8728  0.3437  0.4180     0.3754          0.5977
RF (No Imbalance)             0.8670  0.4325  0.3412     0.2349          0.7955
Random Forest (Weighted)      0.8670  0.4325  0.4257     0.5015          0.7955
CS-GraphSAGE                  0.8577  0.5610  0.5360     0.5321          0.9253
Random Forest                 0.8572  0.4324  0.0701     0.0364          0.7989
GraphSAGE                     0.8567  0.5541  0.5312     0.5242          0.9188
GCN                           0.8360  0.5006  0.4942     0.4885          0.8896
LR (No Imbalance)             0.8253  0.2277  0.1902     0.1097          0.4508
Logistic Regression           0.8247  0.2259  0.3084     0.2593          0.4486
Logistic Regression (SMOTE)   0.8181  0.2309  0.3113     0.2755          0.4746
GAT         